In [22]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from helper import RAGHelper
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
import os
from datasets import load_dataset

import sqlite3
import pandas as pd
import re
import json

import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score
import psycopg

In [23]:
# Constants

DATASET_SOURCE = 'rungalileo/ragbench'
DATASET_NAME = 'tatqa'
DATA_SPLIT = 'test'
# VECTOR_DATABASE = 'chroma'
# GROQ_API_KEY = "GROQ_API_KEY"
TABLE_NAME = 'nextgenrag_questions'
DATABASE_URL = os.getenv('DATABASE_URL')
DOMAIN = 'finance'


In [24]:
dataset = load_dataset(DATASET_SOURCE, DATASET_NAME, split=DATA_SPLIT)

tatqa/train-00000-of-00001.parquet:   0%|          | 0.00/68.9M [00:00<?, ?B/s]

tatqa/validation-00000-of-00001.parquet:   0%|          | 0.00/4.84M [00:00<?, ?B/s]

tatqa/test-00000-of-00001.parquet:   0%|          | 0.00/4.75M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26430 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3336 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3338 [00:00<?, ? examples/s]

In [25]:
try:
    # conn = sqlite3.connect(sqldb)
    conn = psycopg.connect(DATABASE_URL)
    cursor = conn.cursor()
    quest_dict = dict()
    # j = 1
    
    for d in dataset:
        # print(d)
        # print(quest_dict)
        # for ds in dataset:
        #     if ds["question"] == d["question"]:
        if d["question"] in quest_dict:
            if d["generation_model_name"].startswith("gpt"):
                quest_dict[d["question"]].update({
                    "gpt_adherence": d["adherence_score"],
                    "gpt_relevance_score": d["relevance_score"],
                    "gpt_utilization_score": d["utilization_score"],
                    "gpt_completeness_score": d["completeness_score"]
                })
            elif d["generation_model_name"].startswith("claude"):
                quest_dict[d["question"]].update({
                    "claude_adherence": d["adherence_score"],
                    "claude_relevance_score": d["relevance_score"],
                    "claude_utilization_score": d["utilization_score"],
                    "claude_completeness_score": d["completeness_score"]
                })
        else:
            if d["generation_model_name"].startswith("gpt"):
                quest_dict[d["question"]] = {
                    "gpt_adherence": d["adherence_score"],
                    "gpt_relevance_score": d["relevance_score"],
                    "gpt_utilization_score": d["utilization_score"],
                    "gpt_completeness_score": d["completeness_score"]
                }
            elif d["generation_model_name"].startswith("claude"):
                quest_dict[d["question"]] = {
                    "claude_adherence": d["adherence_score"],
                    "claude_relevance_score": d["relevance_score"],
                    "claude_utilization_score": d["utilization_score"],
                    "claude_completeness_score": d["completeness_score"]
        }
                
        # if j == sample:
        #     break
        # else:
        #     j += 1

    # return quest_dict
    questions_count = len(quest_dict.keys())
    i = 1
    # base_session_id = uuid.uuid4()
    for q, v in quest_dict.items():
        # print(q)

        # response, sent_list = self.simple_rag(query=q, expert_domain=domain,
        #                                      retriever=retriever, gen_model=gen_model,
        #                                      temperature=temperature)


        domain_name = DOMAIN 
        dataset_name = DATASET_NAME
        query = q
        gpt_adherence = v.get("gpt_adherence")
        gpt_relevance = v.get("gpt_relevance_score")
        gpt_utilization = v.get("gpt_utilization_score")
        gpt_completeness = v.get("gpt_completeness_score")
        claude_adherence = v.get("claude_adherence")
        claude_relevance = v.get("claude_relevance_score")
        claude_utilization = v.get("claude_utilization_score")
        claude_completeness = v.get("claude_completeness_score") 

        sql_query = f"""
            INSERT INTO {TABLE_NAME} (
                domain_name, dataset_name, query, gpt_adherence, gpt_relevance, gpt_utilization,
                gpt_completeness, claude_adherence, claude_relevance, claude_utilization, claude_completeness
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        values = (
        domain_name,
        dataset_name,
        query,
        gpt_adherence,
        gpt_relevance,
        gpt_utilization,
        gpt_completeness,
        claude_adherence,
        claude_relevance,
        claude_utilization,
        claude_completeness
        )
        cursor.execute(sql_query, values)
        conn.commit()
        print(f"{i}/{questions_count} completed")
        i += 1
        # time.sleep(10)

    conn.close()

    print(f"All entries inserted")

except Exception as e:
    print(e)
    conn.close()

1/1652 completed
2/1652 completed
3/1652 completed
4/1652 completed
5/1652 completed
6/1652 completed
7/1652 completed
8/1652 completed
9/1652 completed
10/1652 completed
11/1652 completed
12/1652 completed
13/1652 completed
14/1652 completed
15/1652 completed
16/1652 completed
17/1652 completed
18/1652 completed
19/1652 completed
20/1652 completed
21/1652 completed
22/1652 completed
23/1652 completed
24/1652 completed
25/1652 completed
26/1652 completed
27/1652 completed
28/1652 completed
29/1652 completed
30/1652 completed
31/1652 completed
32/1652 completed
33/1652 completed
34/1652 completed
35/1652 completed
36/1652 completed
37/1652 completed
38/1652 completed
39/1652 completed
40/1652 completed
41/1652 completed
42/1652 completed
43/1652 completed
44/1652 completed
45/1652 completed
46/1652 completed
47/1652 completed
48/1652 completed
49/1652 completed
50/1652 completed
51/1652 completed
52/1652 completed
53/1652 completed
54/1652 completed
55/1652 completed
56/1652 completed
5

In [26]:
print("hi")

hi
